In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
import random
import os
import pickle
import tensorflow as tf
from tensorflow import keras
from keras.preprocessing.image import ImageDataGenerator
from keras.models import load_model
from tensorflow.keras import layers
from keras.callbacks import EarlyStopping,ModelCheckpoint

In [11]:
DATASET = './datasets/dmd/kaggle_ddd'
IMG_SIZE =(227, 227)
INPUT_SHAPE =  IMG_SIZE + (3,)
BATCH_SIZE = 32
MODEL_SAVE_FOLDER = './models/cnn'
EPOCHS = 30

In [12]:
#data augmetation
data_generate_training = ImageDataGenerator (rescale=1./255,
                                       shear_range = 0.2,
                                       zoom_range = 0.2,
                                       fill_mode = "nearest",
                                       horizontal_flip = True,
                                       width_shift_range = 0.2,
                                       height_shift_range = 0.2,
                                       validation_split = 0.15)

data_generate_test = ImageDataGenerator(rescale = 1./255)

In [13]:
traind = data_generate_training.flow_from_directory(DATASET,
                                                    target_size = IMG_SIZE,
                                                    seed = 123,
                                                    batch_size = BATCH_SIZE,
                                                    subset = "training")
testd = data_generate_training.flow_from_directory(DATASET,
                                                   target_size = IMG_SIZE,
                                                   seed = 123,
                                                   batch_size = BATCH_SIZE,
                                                   subset = "validation")

Found 35525 images belonging to 2 classes.
Found 6268 images belonging to 2 classes.


In [14]:
if os.path.exists(MODEL_SAVE_FOLDER):
    model = keras.models.load_model(MODEL_SAVE_FOLDER + 'best.h5')
else:
    CNNmodel = keras.Sequential([
        layers.Conv2D(32, (3, 3), input_shape=(227, 227, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.5),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.5),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.5),

        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(128, activation = 'relu', kernel_regularizer='l1'),
        layers.Dense(2, activation = 'sigmoid')
    ])

    CNNmodel.compile(optimizer='adam',
                     loss="binary_crossentropy",
                     metrics=['accuracy'])

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=3)

    model_checkpoint = ModelCheckpoint(
        filepath=MODEL_SAVE_FOLDER + 'best.h5',
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        verbose=1
    )

    history = CNNmodel.fit(traind, epochs = 20, validation_data = testd, callbacks=[])

    with open(MODEL_SAVE_FOLDER + 'model_history.pkl', 'wb') as file:
        pickle.dump(history.history, file)

with open(MODEL_SAVE_FOLDER + 'model_history.pkl', 'rb') as file:
    history = pickle.load(file)
    print(history)

Epoch 1/20
1111/1111 [==============================] - 308s 273ms/step - loss: 4.7514 - accuracy: 0.8194 - val_loss: 2.0436 - val_accuracy: 0.5849
Epoch 2/20
1111/1111 [==============================] - 302s 272ms/step - loss: 0.3279 - accuracy: 0.9495 - val_loss: 1.3704 - val_accuracy: 0.7422
Epoch 3/20
 645/1111 [================>.............] - ETA: 1:50 - loss: 0.1912 - accuracy: 0.9685

KeyboardInterrupt: 

In [ ]:
best_model = load_model(MODEL_SAVE_FOLDER + 'best.h5')

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(MODEL_SAVE_FOLDER + 'best.tflite', 'wb') as f:
    f.write(tflite_model)

In [ ]:
acc = history['accuracy']
val_acc = history['val_accuracy']

loss = history['loss']
val_loss = history['val_loss']

epochs_range = range(30)

plt.figure(figsize=(8, 8))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()